# Indian Weather & Rainfall Analysis

## 02 - Data Cleaning & Preprocessing

## Objective

### The objective of this notebook is to clean and preprocess the Indian Weather & Rainfall dataset by handling missing values, creating new features, validating the data, and preparing it for Exploratory Data Analysis (EDA).

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/india_weather_rainfall_data.csv")

### Converting date_of_record into datetime format

In [3]:
df["date_of_record"] = pd.to_datetime(df["date_of_record"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 970339 entries, 0 to 970338
Data columns (total 15 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   date_of_record  970339 non-null  datetime64[us]
 1   month           970339 non-null  str           
 2   season          970339 non-null  str           
 3   station_name    970339 non-null  str           
 4   state           970339 non-null  str           
 5   district        970339 non-null  str           
 6   avg_temp        970339 non-null  float64       
 7   min_temp        926441 non-null  float64       
 8   max_temp        859741 non-null  float64       
 9   wind_speed      695895 non-null  float64       
 10  air_pressure    665675 non-null  float64       
 11  elevation       970339 non-null  int64         
 12  latitude        970339 non-null  float64       
 13  longitude       970339 non-null  float64       
 14  rainfall        712785 non-null  float64       

## Feature Engineering

### Extract Year and Month information

In [4]:
df['year'] = df['date_of_record'].dt.year
df['month_no']=df['date_of_record'].dt.month
df["month"] = df["date_of_record"].dt.month_name()
df.head()

,date_of_record,month,season,station_name,state,district,avg_temp,min_temp,max_temp,wind_speed,air_pressure,elevation,latitude,longitude,rainfall,year,month_no
0,2021-01-02,January,Winter,Gulmarg,JK,Baramulla,-2.2,-6.6,-0.8,2.2,1020.0,2652,34.05,74.4,0.1,2021,1
1,2021-01-03,January,Winter,Gulmarg,JK,Baramulla,-3.6,-4.6,-1.8,3.7,1019.5,2652,34.05,74.4,4.4,2021,1
2,2021-01-04,January,Winter,Gulmarg,JK,Baramulla,-3.0,-4.5,-1.1,2.1,1022.0,2652,34.05,74.4,2.3,2021,1
3,2021-01-05,January,Winter,Gulmarg,JK,Baramulla,-3.3,-5.1,-1.2,2.8,1015.6,2652,34.05,74.4,35.0,2021,1
4,2021-01-06,January,Winter,Gulmarg,JK,Baramulla,-3.9,-8.3,-1.0,3.4,1015.3,2652,34.05,74.4,25.5,2021,1


## Create Full State Names

### The dataset stores state names using abbreviations (e.g., JK, PB, TN). For better readability in analysis and visualizations, we create a new column containing the full names of States and Union Territories.

In [5]:
state_map = {
    "AP": "Andhra Pradesh",
    "AR": "Arunachal Pradesh",
    "AS": "Assam",
    "BR": "Bihar",
    "CG": "Chhattisgarh",
    "CT": "Chhattisgarh",
    "GA": "Goa",
    "GJ": "Gujarat",
    "HR": "Haryana",
    "HP": "Himachal Pradesh",
    "JH": "Jharkhand",
    "JK": "Jammu and Kashmir",
    "KA": "Karnataka",
    "KL": "Kerala",
    "LA": "Ladakh",
    "LD": "Lakshadweep",
    "MH": "Maharashtra",
    "ML": "Meghalaya",
    "MN": "Manipur",
    "MP": "Madhya Pradesh",
    "MZ": "Mizoram",
    "NL": "Nagaland",
    "OD": "Odisha",
    "OR": "Odisha",
    "PB": "Punjab",
    "PY": "Puducherry",
    "RJ": "Rajasthan",
    "SK": "Sikkim",
    "TN": "Tamil Nadu",
    "TR": "Tripura",
    "TS": "Telangana",
    "UK": "Uttarakhand",
    "UP": "Uttar Pradesh",
    "WB": "West Bengal",
    "AN": "Andaman and Nicobar Islands",
    "CH": "Chandigarh",
    "DN": "Dadra and Nagar Haveli and Daman and Diu",
    "DD": "Dadra and Nagar Haveli and Daman and Diu",
    "DL": "Delhi"
}

df["state_name"] = df["state"].map(state_map)

In [6]:
df[["state", "state_name"]].drop_duplicates().sort_values("state")

,state,state_name
793657,AN,Andaman and Nicobar Islands
605779,AP,Andhra Pradesh
98984,AR,Arunachal Pradesh
126231,AS,Assam
167162,BR,Bihar
45490,CH,Chandigarh
474930,CT,Chhattisgarh
508902,DD,Dadra and Nagar Haveli and Daman and Diu
85302,DL,Delhi
681060,GA,Goa


In [7]:
df['state'].nunique(),df['state_name'].nunique()

(32, 32)

### Checking Date Range

In [8]:
df["date_of_record"].min(), df["date_of_record"].max()

(Timestamp('2015-01-01 00:00:00'), Timestamp('2025-02-10 00:00:00'))

### Exploring Missing Values

In [9]:
missing_table = pd.DataFrame({
    "Missing Count":df.isna().sum(),
    "Missing %": df.isna().sum()/len(df)*100,
    "Data Type":df.dtypes
})
missing_table = missing_table[missing_table['Missing Count']>0]
missing_table.sort_values("Missing %",ascending=False)

,Missing Count,Missing %,Data Type
air_pressure,304664,31.397687,float64
wind_speed,274444,28.283311,float64
rainfall,257554,26.542683,float64
max_temp,110598,11.397872,float64
min_temp,43898,4.523986,float64


## Handle Missing Values

### Filling Missing Values in Temperature
#### Since temperature varies by location and season, missing values are filled using the median temperature of the same state and month.

In [10]:
df["max_temp"] = (
    df.groupby(["state", "month"])["max_temp"]
      .transform(lambda x: x.fillna(x.median()))
)

df["min_temp"] = (
    df.groupby(["state", "month"])["min_temp"]
      .transform(lambda x: x.fillna(x.median()))
)

### Filling Air Pressure and Wind Speed
#### Missing values are filled using the median values of the corresponding weather station.

In [11]:
df["air_pressure"] = (
    df.groupby("station_name")["air_pressure"]
      .transform(lambda x: x.fillna(x.median()))
)

df["wind_speed"] = (
    df.groupby("station_name")["wind_speed"]
      .transform(lambda x: x.fillna(x.median()))
)

## Check Remaining Missing Values

In [12]:
df.isna().sum()

date_of_record         0
month                  0
season                 0
station_name           0
state                  0
district               0
avg_temp               0
min_temp               0
max_temp               0
wind_speed             0
air_pressure           0
elevation              0
latitude               0
longitude              0
rainfall          257554
year                   0
month_no               0
state_name             0
dtype: int64

## Rainfall Analysis
### Number of Zero Rainfall Records

In [13]:
(df["rainfall"] == 0).sum()

np.int64(366240)

## Rainfall Statistics by State and Month

In [14]:
df.groupby(["state", "month"])["rainfall"].agg(
    count="count",
    median="median",
    mean="mean"
)

count  median       mean
state month                              
AN    April       1026     0.6   2.527875
      August      1270     5.1   9.801102
      December    1140     1.1   6.619123
      February    1004     0.0   2.020020
      January     1348     0.4   2.747626
...                ...     ...        ...
WB    March       2937     0.0   1.332891
      May         3145     0.8   6.185978
      November    2969     0.0   0.454059
      October     3352     0.5   5.963872
      September   3423     6.6  13.309378

[384 rows x 3 columns]

### Groups with No Rainfall Records

In [15]:
rainfall_group = df.groupby(["state", "month"])["rainfall"].agg(
    count="count",
    median="median"
)

rainfall_group[rainfall_group["count"] == 0]

,,count,median
state,month,,


## Fill Missing Rainfall Values

In [16]:
df["rainfall"] = (
    df.groupby(["state", "month"])["rainfall"]
      .transform(lambda x: x.fillna(x.median()))
)

### Verify Missing Rainfall Values

In [17]:
df["rainfall"].isna().sum()

np.int64(0)

## Dataset Validation
### Dataset Shape

In [18]:
df.shape

(970339, 18)

## Duplicate Records

In [19]:
df.duplicated().sum()

np.int64(0)

## Outlier Detection

### Check Invalid Temperature Values

In [20]:
df[df["max_temp"] > 60]

,date_of_record,month,season,station_name,state,district,avg_temp,min_temp,max_temp,wind_speed,air_pressure,elevation,latitude,longitude,rainfall,year,month_no,state_name
145477,2018-04-29,April,Summer,Jaipur / Sanganer,RJ,Jaipur,38.6,30.0,87.0,12.7,1009.6,385,26.8167,75.8,0.0,2018,4,Rajasthan


### Remove Invalid Records

In [21]:
df = df[df["max_temp"] <= 60]

### Verify Dataset Shape

In [22]:
df.shape

(970338, 18)

### Verify Maximum Temperature and Minimum Temperature

In [23]:
df["max_temp"].max(), df["min_temp"].min()

(np.float64(52.4), np.float64(-18.5))

## Final Dataset Validation


In [24]:
df["state"].nunique()

32

In [25]:
df["state"].value_counts()

state
MH    98770
MP    86379
AP    81129
TN    76059
KA    75376
UP    74032
GJ    61988
RJ    56226
WB    44543
BR    41846
OR    41684
KL    41294
AS    36478
HP    19293
JK    18702
AN    16312
PB    14099
HR    13139
GA    11079
LD     9345
DL     7388
PY     7379
TR     6770
ML     6320
CT     4654
MN     3694
CH     3339
AR     3111
NL     2977
MZ     2792
DD     2652
SK     1489
Name: count, dtype: int64

In [26]:
df["district"].nunique()

314

In [27]:
df["station_name"].nunique()

406

In [28]:
df["season"].value_counts()

season
Winter          333244
Monsoon         241027
Summer          234371
Post-monsoon    161696
Name: count, dtype: int64

# Save Clean Dataset

In [29]:
df.to_csv("../data/cleaned_weather_data.csv", index=False)